In [ ]:
import pandas as pd

# Load the raw dataset
df = pd.read_csv("../data/raw/nyc311_raw.csv", low_memory=False)

# Basic shape check
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Check null percentage per column — helps confirm which columns are mostly empty
null_pct = df.isnull().mean().sort_values(ascending=False) * 100
print(null_pct.round(1))

In [ ]:
print("Number of unique complaint types:", df['complaint_type'].nunique())
print("\nTop 40 complaint types by count:")
print(df['complaint_type'].value_counts().head(40))

In [ ]:
# Module mapping based on agency + complaint_type keywords
module_map = {
    "Sanitation": {
        "agency": ["DSNY"],
        "keywords": ["dirty", "sanitation", "garbage", "recycling", "missed collection", "litter"]
    },
    "Water": {
        "agency": ["DEP"],
        "keywords": ["water", "sewer", "leak", "hydrant"]
    },
    "Electricity": {
        "agency": ["DOT", "CON EDISON"],
        "keywords": ["street light", "lamp", "electric", "power"]
    },
    "Roads": {
        "agency": ["DOT"],
        "keywords": ["street condition", "pothole", "sidewalk", "road", "highway", "curb"]
    },
    "Parks": {
        "agency": ["DPR"],
        "keywords": ["park", "tree", "playground"]
    },
    "Environment": {
        "agency": ["DEP", "DOHMH"],
        "keywords": ["air quality", "noise", "asbestos", "pollution", "hazardous"]
    }
}

def assign_module(row):
    text = str(row['complaint_type']).lower()
    agency = str(row['agency']).upper()
    for module, rule in module_map.items():
        if agency in rule["agency"] and any(k in text for k in rule["keywords"]):
            return module
    return None  # doesn't belong to any of our 6 modules

df['module'] = df.apply(assign_module, axis=1)

print(df['module'].value_counts(dropna=False))

In [ ]:
# Columns to keep (from 2.3 analysis)
keep_cols = [
    'unique_key', 'created_date', 'closed_date', 'agency',
    'complaint_type', 'descriptor', 'location_type', 'incident_zip',
    'status', 'borough', 'latitude', 'longitude',
    'open_data_channel_type', 'module'
]

# Filter: only rows that matched one of our 6 modules
df_filtered = df[df['module'].notna()][keep_cols].copy()

print("Filtered shape:", df_filtered.shape)
print("\nModule distribution:")
print(df_filtered['module'].value_counts())

# Save to processed folder
df_filtered.to_csv("../data/processed/nyc311_filtered.csv", index=False)

In [ ]:
print(df_filtered.shape)
df_filtered.head()

In [ ]:
print(df_filtered.shape)

print(df_filtered["module"].value_counts())

df_filtered.head()

In [ ]:
import os

print(os.getcwd())

In [ ]:
print(os.path.exists("data/processed/nyc311_filtered.csv"))
print(os.path.exists("../data/processed/nyc311_filtered.csv"))

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/nyc311_filtered.csv", low_memory=False)
print(df.shape)
df.info()

In [ ]:
import pandas as pd

PROCESSED_DATA = "../data/processed/nyc311_filtered.csv"

df = pd.read_csv(PROCESSED_DATA)

print(df.shape)
print(df["module"].value_counts())
print(df.isnull().mean().sort_values(ascending=False))

In [ ]:
import pandas as pd
raw = pd.read_csv("../data/raw/nyc311_raw.csv", low_memory=False)
print(raw.shape)
raw.head()

In [ ]:
print(df_filtered.shape)
print(df_filtered['module'].value_counts())


In [ ]:
df_filtered.to_csv("../data/processed/nyc311_filtered.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/nyc311_filtered.csv", low_memory=False)

print(df.shape)
df.info()

In [ ]:
print("Total rows:", len(df))
print("Duplicate unique_key rows:", df['unique_key'].duplicated().sum())

df = df.drop_duplicates(subset='unique_key', keep='first')

print("Rows after dedup:", len(df))

In [ ]:
null_summary = df.isnull().sum().sort_values(ascending=False)
print(null_summary[null_summary > 0])


In [ ]:
# Fill categorical fields
df['descriptor'] = df['descriptor'].fillna('Unknown')
df['location_type'] = df['location_type'].fillna('Unspecified')

# Drop rows missing critical geographic fields
before = len(df)
df = df.dropna(subset=['incident_zip', 'latitude', 'longitude'])
after = len(df)

print(f"Dropped {before - after} rows missing zip/lat/long")
print(f"Remaining rows: {after}")

In [ ]:
df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')
df['closed_date'] = pd.to_datetime(df['closed_date'], errors='coerce')

print(df[['created_date', 'closed_date']].dtypes)
print(df[['created_date', 'closed_date']].head())

In [ ]:
# Check the actual range in your data
print(df[['latitude', 'longitude']].describe())

In [ ]:
before = len(df)

df = df[
    (df['latitude'].between(40.4, 40.95)) &
    (df['longitude'].between(-74.3, -73.65))
]

after = len(df)
print(f"Dropped {before - after} rows with invalid coordinates")
print(f"Remaining rows: {after}")

In [ ]:
df = df.rename(columns={
    'incident_zip': 'zip_code',
    'open_data_channel_type': 'submission_channel'
})

print(df.columns.tolist())

In [ ]:
# Check current unique values
print(df['borough'].unique())
print(df['status'].unique())

In [ ]:
# Standardize casing and strip whitespace
df['borough'] = df['borough'].str.strip().str.title()
df['status'] = df['status'].str.strip().str.title()
df['location_type'] = df['location_type'].str.strip().str.title()

print(df['borough'].value_counts())

In [ ]:
print("Final shape:", df.shape)
print("\nRemaining nulls:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nBorough distribution:")
print(df['borough'].value_counts())

In [ ]:
df.to_csv("../data/processed/nyc311_cleaned.csv", index=False)
print("Saved cleaned dataset:", df.shape)

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/processed/nyc311_cleaned.csv", parse_dates=['created_date', 'closed_date'])

print(df.shape)
df.head()

In [ ]:
print(df.dtypes[['created_date', 'closed_date']])

In [ ]:
module_counts = df['module'].value_counts().reset_index()
module_counts.columns = ['module', 'count']

fig = px.bar(
    module_counts,
    x='module', y='count',
    title='Complaint Distribution by Module',
    color='module',
    text='count'
)
fig.update_layout(showlegend=False)
fig.show()

**Insight:** Sanitation (16,096) and Water (15,331) are the two largest categories, followed by Roads and Environment. Electricity is the smallest module (5,166) — this is expected, since NYC 311 does not route power outages through a dedicated agency; most electrical complaints are captured indirectly through DOT streetlight reports. This imbalance should be considered when training classification models in Phase 6.

In [ ]:
borough_counts = df['borough'].value_counts().reset_index()
borough_counts.columns = ['borough', 'count']

fig = px.bar(
    borough_counts,
    x='borough', y='count',
    title='Complaints by Borough',
    color='borough',
    text='count'
)
fig.update_layout(showlegend=False)
fig.show()

**Insight:** Complaint volume roughly tracks real NYC population ranking — Brooklyn (22,248) and Queens (19,317) lead, followed by Manhattan, Bronx, and Staten Island. This consistency confirms the dataset is not skewed by a filtering or scraping error. A small "Unspecified" borough category (6 rows) exists and is treated as negligible noise.

In [ ]:
df['date_only'] = df['created_date'].dt.date
daily_counts = df.groupby('date_only').size().reset_index(name='count')

fig = px.line(
    daily_counts,
    x='date_only', y='count',
    title='Daily Complaint Volume Over Time'
)
fig.show()

**Insight:** Complaint volume shows a clear repeating weekly cycle — regular peaks and dips roughly every 7 days — most likely reflecting lower complaint activity on weekends. This supports adding a `day_of_week` / `is_weekend` feature in Phase 5.

In [ ]:
df['year_month'] = df['created_date'].dt.to_period('M').astype(str)
monthly = df.groupby(['year_month', 'module']).size().reset_index(name='count')

fig = px.line(
    monthly,
    x='year_month', y='count', color='module',
    title='Monthly Complaint Trends by Module',
    markers=True
)
fig.show()

**Insight:** Water and Electricity complaints show the steepest upward trend across the observed months, while Environment and Roads grow more gradually. Note: this dataset spans roughly Oct 2024–Jan 2025 (a few months), not a full multi-year range, so seasonal patterns (e.g., summer vs. winter) cannot be fully assessed from this extract alone.

In [ ]:
# Only use resolved complaints (closed_date not null)
resolved = df.dropna(subset=['closed_date']).copy()
resolved['response_hours'] = (resolved['closed_date'] - resolved['created_date']).dt.total_seconds() / 3600

# Filter out negative/absurd values (data entry errors)
resolved = resolved[(resolved['response_hours'] >= 0) & (resolved['response_hours'] <= 24*90)]  # cap at 90 days

agency_perf = resolved.groupby('agency')['response_hours'].median().sort_values().reset_index()

fig = px.bar(
    agency_perf,
    x='agency', y='response_hours',
    title='Median Response Time by Agency (Hours)'
)
fig.show()

**Insight:** DOHMH resolves complaints fastest (~12 hours median), while DOT and DPR are slowest (~65+ hours). This makes sense — health-related complaints (DOHMH) are often quick administrative responses, while road (DOT) and park (DPR) issues typically require physical repair work.

In [ ]:
fig = px.histogram(
    resolved,
    x='response_hours',
    nbins=100,
    title='Distribution of Response Times (Hours)'
)
fig.show()

Response time is heavily right-skewed — most complaints resolve within a few days, but a long tail extends to 60-90+ days. This will require a log-transformation of the target variable for response-time prediction models in Phase 6.

In [ ]:
df = df[
    (df['latitude'].between(40.49, 40.92)) &
    (df['longitude'].between(-74.26, -73.68))
]
print(df.shape)

In [ ]:
df.to_csv("../data/processed/nyc311_cleaned.csv", index=False)

In [ ]:
print("Before:", df.shape)

# Filter 1: keep only valid NYC ZIP codes (already applied, kept for clarity)
nyc_zip_ranges = [
    (10001, 10282), (10301, 10314), (10451, 10475),
    (11004, 11109), (11201, 11256), (11351, 11697),
]
df['zip_code'] = df['zip_code'].astype(int)
df = df[df['zip_code'].apply(lambda z: any(low <= z <= high for low, high in nyc_zip_ranges))]

# Filter 2: keep only coordinates within NYC's actual bounding box
df = df[
    (df['latitude'].between(40.49, 40.92)) &
    (df['longitude'].between(-74.26, -73.68))
]

print("After:", df.shape)

df.to_csv("../data/processed/nyc311_cleaned.csv", index=False)
print("Saved.")

In [ ]:
sample = df.sample(n=10000, random_state=42)

fig = px.scatter_mapbox(
    sample,
    lat='latitude', lon='longitude',
    color='borough',
    hover_data=['complaint_type', 'module'],
    zoom=9,
    height=700,
    title='Complaint Hotspots by Borough'
)
fig.update_layout(mapbox_style="open-street-map")
fig.show()

In [ ]:
cross = df.groupby(['borough', 'module']).size().reset_index(name='count')

fig = px.bar(
    cross,
    x='borough', y='count', color='module',
    title='Complaints by Borough and Module',
    barmode='stack'
)
fig.show()

Manhattan shows a disproportionately high share of Environment (noise) complaints relative to its total volume, likely reflecting its density and nightlife activity. Staten Island shows a comparatively higher share of Water complaints, possibly linked to older infrastructur

## Phase 4 Summary

Key takeaways carried into Phase 5 (Feature Engineering):
- Weekly cyclical pattern in complaint volume → build `day_of_week` and `is_weekend` features
- Response time is right-skewed → apply log-transform before modeling
- Module distribution is imbalanced (Electricity is smallest) → consider this when evaluating classification model performance
- Borough and module patterns differ meaningfully → borough and module are both strong candidate features
- Geographic clustering is visually distinct by borough → borough itself may be sufficient as a geographic feature, reducing the need for complex geo-clustering

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/nyc311_cleaned.csv", parse_dates=['created_date', 'closed_date'])
print(df.shape)
df.head()

In [ ]:
print(df.dtypes[['created_date', 'closed_date']])

In [ ]:
df['response_hours'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 3600

print(df['response_hours'].describe())

In [ ]:
invalid_response = (df['response_hours'] < 0).sum()
print(f"Rows with negative response time: {invalid_response}")

# Negative response time is a data error (closed before created) — remove these
df = df[(df['response_hours'] >= 0) | (df['response_hours'].isna())]
print(df.shape)

In [ ]:
# Cap at 90 days, consistent with EDA
df['response_hours_capped'] = df['response_hours'].clip(upper=90*24)

In [ ]:
df['response_hours_log'] = np.log1p(df['response_hours_capped'])

print(df[['response_hours', 'response_hours_capped', 'response_hours_log']].describe())

In [ ]:
import plotly.express as px
fig = px.histogram(df, x='response_hours_log', nbins=50, title='Log-Transformed Response Time')
fig.show()

In [ ]:
df['is_resolved'] = df['closed_date'].notna().astype(int)

print(df['is_resolved'].value_counts())

In [ ]:
df['day_of_week'] = df['created_date'].dt.day_name()
df['day_of_week_num'] = df['created_date'].dt.dayofweek  # 0=Monday, 6=Sunday

print(df['day_of_week'].value_counts())

In [ ]:
df['is_weekend'] = df['day_of_week_num'].isin([5, 6]).astype(int)

print(df['is_weekend'].value_counts())

In [ ]:
df['hour_of_day'] = df['created_date'].dt.hour

print(df['hour_of_day'].value_counts().sort_index())

In [ ]:
fig = px.histogram(df, x='hour_of_day', nbins=24, title='Complaints by Hour of Day')
fig.show()

In [ ]:
df['month'] = df['created_date'].dt.month

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['season'] = df['month'].apply(get_season)

print(df['season'].value_counts())

In [ ]:
high_priority_keywords = ['emergency', 'gas', 'fire', 'water main', 'structural', 'collapse']
medium_priority_keywords = ['noise', 'illegal', 'blocked', 'leak']

def assign_priority(complaint_type):
    text = str(complaint_type).lower()
    if any(k in text for k in high_priority_keywords):
        return 'High'
    elif any(k in text for k in medium_priority_keywords):
        return 'Medium'
    else:
        return 'Low'

df['priority'] = df['complaint_type'].apply(assign_priority)

print(df['priority'].value_counts())

In [ ]:
print(df['module'].value_counts())

In [ ]:
# One-hot encode low-cardinality categoricals
categorical_cols = ['borough', 'module', 'priority', 'season', 'submission_channel']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(df_encoded.shape)
print(df_encoded.columns.tolist())

In [ ]:
final_features = [
    'unique_key', 'created_date', 'closed_date', 'complaint_type',
    'response_hours', 'response_hours_capped', 'response_hours_log',
    'is_resolved', 'day_of_week', 'day_of_week_num', 'is_weekend',
    'hour_of_day', 'month', 'season', 'priority', 'module', 'borough',
    'latitude', 'longitude', 'zip_code', 'status', 'agency'
]

df_final = df[final_features].copy()

print(df_final.shape)
df_final.head()

df_final.to_csv("../data/processed/nyc311_features.csv", index=False)
print("Saved feature-engineered dataset.")

In [ ]:
print(df_final.isnull().sum())

In [ ]:
import pandas as pd
df = pd.read_csv("../data/processed/nyc311_features.csv")
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())